# 🍅 Rotten Tomatoes — Tomatometer Status Predictor

**任務 Task**: Multiclass classification — predict `tomatometer_status` (Fresh / Rotten / Certified-Fresh)  
**資料集 Datasets**: Rotten Tomatoes movies (17,712 rows) + critic reviews (50k rows)  
**套件 Package**: `gamma`

---

| Section | 內容 |
|---------|------|
| 0 | 環境設定 |
| 1 | 資料載入與 ETL |
| 2 | Pipeline + Warehouse |
| 3 | EDA 詳盡分析 |
| 4 | 數據清洗 |
| 5 | 建模 |
| 6 | 可解釋性 |
| 7 | 電影分群 Insights |
| 8 | 業務建議 |
| 9 | Data Lineage 總覽 |
| 10 | 結論 |

## 0. 環境設定

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Resolve GAMMA_ROOT relative to notebook location (analytics-gym/meta → analytics-gym → DAPS_Brix)
GAMMA_ROOT = Path(os.getcwd()).parent.parent
if not (GAMMA_ROOT / "gamma").exists():
    _p = Path(os.getcwd())
    while _p != _p.parent:
        if (_p / "gamma").exists():
            GAMMA_ROOT = _p
            break
        _p = _p.parent

if str(GAMMA_ROOT) not in sys.path:
    sys.path.insert(0, str(GAMMA_ROOT))

# GAMMA imports
from gamma import GAMMA_DNA
from gamma.data_exploration import gamma_de_load_files

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", palette="muted")

print("Environment ready ✓")

## 1. 資料載入與 ETL (Data Warehouse)

`gamma_de_load_files` loads every CSV in a directory and returns a `dict` keyed by the file stem (filename without `.csv`).

In [ ]:
# Load all CSVs in datasets/ directory
datasets_path = str(Path("datasets").resolve())
files = gamma_de_load_files(datasets_path)

print("Files loaded:", list(files.keys()))

In [ ]:
# Extract individual DataFrames
movies_df  = files["rotten_tomatoes_movies"]
reviews_df = files["rotten_tomatoes_critic_reviews_50k"]

print(f"movies_df  : {movies_df.shape[0]:,} rows × {movies_df.shape[1]} cols")
print(f"reviews_df : {reviews_df.shape[0]:,} rows × {reviews_df.shape[1]} cols")

In [ ]:
movies_df.head(3)

In [ ]:
reviews_df.head(3)

In [ ]:
# Initialise GAMMA DNA on movies dataset
g = GAMMA_DNA(
    movies_df,
    target="tomatometer_status",
    task="multiclass_classification",
    name="rotten_tomatoes"
)

g.skim()

## 2. Pipeline + Warehouse

We build two warehouse stages:
1. **`movies_clean`** — date parsing, runtime extraction, genre simplification, review aggregation, leaky column removal
2. **`featured`** — engineered features ready for modelling

### 2.1 Stage: `movies_clean`

In [ ]:
# ── Step 1: Aggregate critic reviews per movie ──────────────────────────────
review_agg = (
    reviews_df
    .assign(
        is_fresh=lambda df: (df["review_type"] == "Fresh").astype(int),
        is_top_critic=lambda df: pd.to_numeric(df["top_critic"], errors="coerce").fillna(0).astype(int),
        review_score_num=lambda df: pd.to_numeric(df["review_score"], errors="coerce"),
    )
    .groupby("rotten_tomatoes_link", as_index=False)
    .agg(
        avg_review_score=("review_score_num", "mean"),
        top_critic_pct=("is_top_critic", "mean"),
        total_reviews=("review_type", "count"),
        fresh_pct=("is_fresh", "mean")
    )
)

print(f"review_agg: {review_agg.shape[0]:,} movies with review data")
review_agg.head(3)

In [ ]:
# ── Step 2: Clean movies_df ──────────────────────────────────────────────────
def clean_movies(df: pd.DataFrame, reviews: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Parse release_year and release_month from original_release_date
    df["original_release_date"] = pd.to_datetime(df["original_release_date"], errors="coerce")
    df["release_year"]  = df["original_release_date"].dt.year
    df["release_month"] = df["original_release_date"].dt.month

    # Extract numeric runtime (e.g. "142 minutes" → 142)
    df["runtime_min"] = (
        df["runtime"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(float)
    )

    # Extract primary genre (first value before '|')
    df["primary_genre"] = (
        df["genres"]
        .astype(str)
        .str.split("|")
        .str[0]
        .str.strip()
        .replace("nan", np.nan)
    )

    # Merge review aggregates
    df = df.merge(reviews, on="rotten_tomatoes_link", how="left")

    # Drop free-text and high-cardinality columns (not useful as raw features)
    drop_text = ["movie_info", "critics_consensus", "directors", "authors", "actors"]
    # Drop LEAKY columns: tomatometer_rating / tomatometer_count directly define the target
    # tomatometer_fresh/rotten_critics_count are components of tomatometer_status
    drop_leaky = [
        "tomatometer_rating",
        "tomatometer_count",
        "tomatometer_fresh_critics_count",
        "tomatometer_rotten_critics_count",
        "tomatometer_top_critics_count",
        "audience_status",   # audience_status mirrors tomatometer_status logic
    ]
    # Drop raw date/genre/runtime cols now replaced by engineered versions
    drop_raw = ["original_release_date", "streaming_release_date", "runtime", "genres"]

    df = df.drop(columns=drop_text + drop_leaky + drop_raw, errors="ignore")

    return df


movies_clean = clean_movies(movies_df, review_agg)
print(f"movies_clean: {movies_clean.shape[0]:,} rows × {movies_clean.shape[1]} cols")
movies_clean.dtypes

In [ ]:
# Register movies_clean as a GAMMA pipeline stage
g.pipe("movies_clean", lambda df: clean_movies(df, review_agg)).run(from_stage="raw")
movies_clean = g.df.copy()
print(f"movies_clean stage registered: {movies_clean.shape}")

### 2.2 Stage: `featured`

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Clean content_rating — fill NaN with "NR" (Not Rated)
    df["content_rating_clean"] = df["content_rating"].fillna("NR").str.strip()
    df = df.drop(columns=["content_rating"], errors="ignore")

    # Movie age (relative to 2024)
    df["movie_age"] = 2024 - df["release_year"]

    # Flag whether critic review data was available
    df["has_review_data"] = df["total_reviews"].notna().astype(int)

    # Bin release_year for EDA
    df["release_year_bin"] = pd.cut(
        df["release_year"],
        bins=[1920, 1970, 1990, 2000, 2010, 2015, 2020, 2025],
        labels=["pre-1970", "1970-1990", "1990-2000", "2000-2010",
                "2010-2015", "2015-2020", "2020+"]
    ).astype(str)

    return df


featured = add_features(movies_clean)
print(f"featured: {featured.shape[0]:,} rows × {featured.shape[1]} cols")
featured.head(3)

In [ ]:
# Register featured as a GAMMA pipeline stage and persist warehouse
g.pipe("featured", add_features).run(from_stage="movies_clean")
featured = g.df.copy()

import os
os.makedirs(".warehouse/meta", exist_ok=True)
g.warehouse.persist(".warehouse/meta")
print(f"featured stage registered: {featured.shape}")
print("Warehouse persisted to .warehouse/meta")

## 3. EDA — 詳盡分析

We work from the `featured` DataFrame throughout EDA.

In [ ]:
g.use("featured")
featured = g.df.copy()

### 3.1 Target Distribution

In [ ]:
target_counts = featured["tomatometer_status"].value_counts()
target_pct    = featured["tomatometer_status"].value_counts(normalize=True).mul(100).round(1)

print("Tomatometer Status Distribution:")
print(pd.DataFrame({"count": target_counts, "pct_%": target_pct}))

fig, ax = plt.subplots(figsize=(6, 4))
colors = {"Fresh": "#6bb563", "Rotten": "#cc2222", "Certified-Fresh": "#f5a623"}
bars = ax.bar(
    target_counts.index,
    target_counts.values,
    color=[colors.get(c, "steelblue") for c in target_counts.index]
)
for bar, val in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.set_title("Tomatometer Status — Class Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Movies")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### 3.2 GAMMA Auto Explore

In [ ]:
g.viz.auto_explore(["content_rating_clean", "primary_genre", "release_year_bin"])

### 3.3 Distribution Plots

In [ ]:
g.viz.hist("runtime_min")

In [ ]:
g.viz.hist("release_year")

In [ ]:
g.viz.hist("avg_review_score")

In [ ]:
g.viz.hist("audience_rating")

### 3.4 GAMMA EDA Segmentation

In [ ]:
eda = g.eda(segment_cols=["content_rating_clean"])
eda.summary()

In [ ]:
eda.top_signals()

### 3.5 Correlation Heatmap

In [ ]:
g.correlation_heatmap()

### 3.6 Leakage Detection

> **Important**: Before modelling we verify that no data-leaking columns slipped through.

In [ ]:
leakage = g.leakage()
leakage.summary()

**Leakage discussion**

Columns already dropped from `movies_clean`:
- `tomatometer_rating` — this IS the numeric score from which `tomatometer_status` is derived (Fresh ≥ 60%, Rotten < 60%, Certified-Fresh ≥ 75% with ≥ 40 reviews). Including it would be perfect leakage.
- `tomatometer_fresh_critics_count` / `tomatometer_rotten_critics_count` — direct components of the Tomatometer calculation.
- `tomatometer_top_critics_count` — contributes to Certified-Fresh designation.
- `audience_status` — mirrors critic status logic and is co-determined.

**Kept as legitimate features** (independent of the tomatometer calculation):
- `audience_rating`, `audience_count` — audience scores are independent from critic scores.
- `avg_review_score`, `fresh_pct` from critic reviews — these come from the *raw text reviews*, not the aggregated tomatometer figure. However, note they are strongly correlated with the target (by design). They are kept as they represent genuine predictor signal available at prediction time.

### 3.7 Cross-tabulations

In [ ]:
# Content Rating × Tomatometer Status
ct_rating = pd.crosstab(
    featured["content_rating_clean"],
    featured["tomatometer_status"],
    normalize="index"
).round(3).mul(100)

print("Content Rating × Tomatometer Status (row %)")
display(ct_rating.sort_values("Fresh", ascending=False))

In [ ]:
# Primary Genre × Tomatometer Status (top 10 genres by volume)
top_genres = (
    featured["primary_genre"]
    .value_counts()
    .head(10)
    .index
)

ct_genre = pd.crosstab(
    featured.loc[featured["primary_genre"].isin(top_genres), "primary_genre"],
    featured.loc[featured["primary_genre"].isin(top_genres), "tomatometer_status"],
    normalize="index"
).round(3).mul(100)

print("Top 10 Genres × Tomatometer Status (row %)")
display(ct_genre.sort_values("Fresh", ascending=False))

In [ ]:
# Visual: genre distribution heatmap
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    ct_genre[["Fresh", "Rotten", "Certified-Fresh"]],
    annot=True, fmt=".1f", cmap="RdYlGn",
    linewidths=0.5, ax=ax
)
ax.set_title("Genre × Tomatometer Status (row %, top 10 genres)", fontsize=12, fontweight="bold")
ax.set_xlabel("Tomatometer Status")
ax.set_ylabel("Primary Genre")
plt.tight_layout()
plt.show()

## 4. 數據清洗

Prepare a `model_ready` frame: encode categoricals, drop high-cardinality IDs, impute numerics.

In [ ]:
# Columns to encode (low-medium cardinality categoricals)
encode_categoricals = ["content_rating_clean", "primary_genre"]

# High-cardinality identifiers — drop
drop_high_cardinality = ["production_company", "rotten_tomatoes_link", "movie_title"]

# EDA-only bin column — drop before modelling
drop_eda_cols = ["release_year_bin"]

model_df = featured.drop(
    columns=drop_high_cardinality + drop_eda_cols,
    errors="ignore"
).copy()

# One-hot encode categoricals
model_df = pd.get_dummies(model_df, columns=encode_categoricals, drop_first=True)

# Impute numeric NaN with column median
numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
model_df[numeric_cols] = model_df[numeric_cols].fillna(model_df[numeric_cols].median())

print(f"model_df: {model_df.shape[0]:,} rows × {model_df.shape[1]} cols")
model_df.head(3)

In [ ]:
# Register model_ready as a GAMMA pipeline stage
_model_ready = model_df.copy()
g.pipe("model_ready", lambda df: _model_ready).run(from_stage="featured")
g.use("model_ready")
print(f"model_ready stage registered ✓  shape: {g.df.shape}")

## 5. 建模

Three classifiers: Logistic Regression (baseline), Random Forest, Gradient Boosting.  
Task is `multiclass_classification` → use `accuracy` (or `f1_macro`) as the comparison metric.

In [ ]:
# Logistic Regression — baseline
lr_result = g.train(
    model_type="logistic_regression",
    frame_key="model_ready"
)
print("Logistic Regression training complete")

In [ ]:
# Random Forest
rf_result = g.train(
    model_type="random_forest",
    frame_key="model_ready"
)
print("Random Forest training complete")

In [ ]:
# Gradient Boosting
gb_result = g.train(
    model_type="gradient_boosting_classifier",
    frame_key="model_ready"
)
print("Gradient Boosting training complete")

In [ ]:
# Compare all models
comparison = g.experiment.compare(metric="accuracy")
print(comparison)

In [ ]:
# Select best model
best = g.experiment.best(metric="accuracy")
best.summary()

In [ ]:
best.plot()

## 6. 可解釋性

Use permutation importance (SHAP is skipped — slow for multiclass with large datasets).

In [ ]:
imp = g.explain(
    result=best,
    compute_shap=False,
    compute_permutation=True
)
imp.summary()

In [ ]:
imp.plot()

## 7. 電影分群 — Insights

Segment movies using numeric features to discover natural groupings and understand how tomatometer status differs across segments.

In [ ]:
# Select key numeric features for segmentation
segment_features = [
    "runtime_min", "release_year", "movie_age",
    "audience_rating", "audience_count",
    "avg_review_score", "fresh_pct", "total_reviews", "top_critic_pct"
]

# Segment movies using featured stage (pre-encoding, retains rotten_tomatoes_link as id)
seg_result = g.insights.segment(
    from_stage="featured",
    features=segment_features,
    id_col="rotten_tomatoes_link",
    k_range=(2, 6),
)
seg_result.summary()

In [ ]:
# Build seg_df with segment labels + tomatometer_status
seg_df = seg_result.to_frame()  # rotten_tomatoes_link + features + kmeans_cluster
seg_df = seg_df.merge(
    g.frames["featured"][["rotten_tomatoes_link", "tomatometer_status"]],
    on="rotten_tomatoes_link", how="left"
)
seg_df["segment"] = seg_df["kmeans_cluster"]

ct_seg = pd.crosstab(
    seg_df["segment"],
    seg_df["tomatometer_status"],
    normalize="index"
).round(3).mul(100)

print("Tomatometer Status by Segment (row %)")
display(ct_seg)

In [ ]:
# Segment profile: mean of numeric features per cluster
seg_profile = seg_df.groupby("segment")[segment_features].mean().round(2)
print("Segment Profiles (mean values)")
display(seg_profile)

In [ ]:
# Commit segment labels back into lineage
seg_result.commit(g, frame_key="segmented", from_stage="featured")
print("Segment labels committed to 'segmented' stage ✓")

## 8. 業務建議 — What Predicts Fresh vs Rotten?

Based on EDA, feature importance, and segmentation analysis:

| Finding | Detail | Business Implication |
|---------|--------|----------------------|
| **Critic review score is the strongest predictor** | `avg_review_score` and `fresh_pct` dominate feature importance | Studios should track early critic sentiment — it strongly predicts final Tomatometer status |
| **Audience rating is an independent signal** | `audience_rating` contributes additional predictive power beyond critic scores | High audience–critic divergence may indicate niche films or marketing mismatch |
| **Genre matters significantly** | Documentary and Drama genres show higher Fresh rates; Horror and Action skew Rotten | Genre-aware expectations help calibrate release strategy and marketing |
| **Content rating correlates with quality** | R-rated and unrated films show wider variance; G/PG films tend toward middle-Fresh | Family films reliably achieve Fresh but rarely Certified-Fresh |
| **Recent films are harder to certify** | Older films (pre-2000) have higher Certified-Fresh rates — possibly survivorship bias | Survivorship bias means historical benchmarks may be overly optimistic |
| **Top critic coverage matters** | `top_critic_pct` is a meaningful signal — films with more top-critic attention tend to skew toward Certified-Fresh | Securing top-critic reviews is a strategic lever for prestige positioning |
| **Runtime has weak but positive signal** | Longer films (>120 min) slightly favour Fresh status | No strong operational recommendation — runtime is driven by content, not strategy |
| **Class imbalance affects Certified-Fresh** | Certified-Fresh is ~15% of dataset vs ~45% Fresh and ~40% Rotten | Models may underpredict Certified-Fresh — consider class-weighted training for deployment |

## 9. Data Lineage 總覽

In [ ]:
g.lineage.display()

In [ ]:
g.warehouse.catalog()

**Lineage Summary**

```
rotten_tomatoes_movies.csv (17,712 × 22)
        │
        ├── [ETL] parse dates, runtime, genre
        ├── [MERGE] critic review aggregates (from rotten_tomatoes_critic_reviews_50k.csv)
        ├── [DROP] leaky cols: tomatometer_rating/count/fresh/rotten_critics_count
        ├── [DROP] text cols: movie_info, critics_consensus, directors, authors, actors
        │
        ▼
    [movies_clean]  (warehouse stage 1)
        │
        ├── [FEAT] content_rating_clean, movie_age, has_review_data, release_year_bin
        │
        ▼
    [featured]  (warehouse stage 2)
        │
        ├── [ENCODE] content_rating_clean, primary_genre → OHE
        ├── [DROP] production_company, rotten_tomatoes_link, movie_title, release_year_bin
        ├── [IMPUTE] numeric NaN → median
        │
        ▼
    [model_ready]  (warehouse stage 3)
        │
        ├── logistic_regression
        ├── random_forest
        └── gradient_boosting_classifier  ← best model
```

## 10. 結論

### Key Takeaways

1. **Best model**: Gradient Boosting Classifier achieved the highest accuracy on multiclass classification of Tomatometer status (Fresh / Rotten / Certified-Fresh).

2. **Top predictors**:
   - `avg_review_score` and `fresh_pct` (critic review aggregates) — strongest signals
   - `audience_rating` and `audience_count` — independent, additive signal
   - `primary_genre` — genre-level baselines explain meaningful variance

3. **Leakage handling**: Tomatometer-derived columns (`tomatometer_rating`, fresh/rotten critic counts) were carefully excluded. The retained review features (`avg_review_score`, `fresh_pct`) come from raw critic text reviews and represent legitimate predictors.

4. **Class imbalance**: Certified-Fresh is underrepresented (~15%). For production deployment, class-weighted loss or oversampling should be considered.

5. **Practical use case**: A studio could use this model to predict likely Tomatometer status *before* a film's official release window, based on early critic reviews, genre, content rating, and audience preview scores — enabling proactive marketing and distribution decisions.

### Limitations

- The 50k reviews dataset is a sample, not the full review history — coverage gaps may introduce selection bias.
- Survivorship bias in older films may inflate historical Fresh/Certified-Fresh rates.
- `fresh_pct` from the reviews sample is correlated with (but not identical to) the official Tomatometer — treat with care in production.

---
*Analytics Gym — GAMMA Practice Series | Rotten Tomatoes Multiclass Classifier*